# T01 Dataset Verification

This notebook verifies the structure and proposal claims for `data/raw/weatherAUS.csv`. It performs no EDA, preprocessing, feature engineering, row removal, imputation, or modelling. The raw dataset is read only.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fdm_rainfall.data import load_weather_data
from fdm_rainfall.validation import verify_weather_dataset

DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'weatherAUS.csv'
TABLES_DIR = PROJECT_ROOT / 'reports' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset path: {DATASET_PATH.relative_to(PROJECT_ROOT)}')

Dataset path: data\raw\weatherAUS.csv


## Load the dataset

Loading uses the reusable project function and does not write to or alter the source CSV.

In [2]:
weather = load_weather_data(DATASET_PATH)
print(f'Loaded successfully: {weather.shape[0]:,} rows x {weather.shape[1]} columns')

Loaded successfully: 145,460 rows x 23 columns


## Required verification findings

In [3]:
verification = verify_weather_dataset(weather)
summary_table = verification.summary_table()
display(summary_table)

,Verification item,Actual dataset value
0,Dataset loaded successfully,True
1,Total rows,145460
2,Total columns,23
3,Date minimum,2007-11-01
4,Date maximum,2017-06-25
5,Invalid or missing dates,0
6,Unique Locations,49
7,RainTomorrow exists,True
8,RainTomorrow unique non-missing values,"No, Yes"
9,RainTomorrow missing count,3267


## Exact columns and inferred data types

In [4]:
column_schema_table = verification.column_schema_table()
display(column_schema_table)

,Position,Column,Pandas dtype
0,1,Date,str
1,2,Location,str
2,3,MinTemp,float64
3,4,MaxTemp,float64
4,5,Rainfall,float64
5,6,Evaporation,float64
6,7,Sunshine,float64
7,8,WindGustDir,str
8,9,WindGustSpeed,float64
9,10,WindDir9am,str


## RainTomorrow values and class counts

Counts are reported exactly as loaded. Missing target values are shown separately; no rows are removed and no values are imputed.

In [5]:
target_counts_table = verification.target_counts_table()
display(target_counts_table)

,RainTomorrow value,Count
0,No,110316
1,Yes,31877
2,<MISSING>,3267


## Proposal-verification table

Every supplied proposal claim is compared directly with the loaded dataset. Mismatches, if any, remain visible and are not corrected silently.

In [6]:
proposal_table = verification.proposal_comparison()
display(proposal_table)

,Claim,Expected value,Actual dataset value,Match / Mismatch,Comment
0,Total rows,145460,145460,Match,Compares all loaded observations.
1,Total columns,23,23,Match,Compares the complete CSV header width.
2,Unique locations,49,49,Match,Counts distinct non-missing Location values.
3,Labelled RainTomorrow rows,142193,142193,Match,Counts non-missing target values.
4,Missing RainTomorrow rows,3267,3267,Match,Counts missing target values.
5,Date range,2007-11-01 to 2017-06-25,2007-11-01 to 2017-06-25,Match,Uses parsed valid dates and inclusive endpoints.
6,RISK_MM exists,False,False,Match,Leakage-prone RISK_MM is expected to be absent.


## Save verification tables

Only derived verification summaries are written to `reports/tables/`; the raw dataset remains unchanged.

In [7]:
column_names_table = pd.DataFrame({
    'Position': range(1, len(verification.column_names) + 1),
    'Column': verification.column_names,
})
duplicate_table = pd.DataFrame(
    [
        ('Exact duplicate rows (excluding first occurrence)', verification.exact_duplicate_rows),
        ('Duplicate Date + Location combinations', verification.duplicate_date_location_combinations),
        ('Rows in duplicate Date + Location combinations', verification.duplicate_date_location_rows),
    ],
    columns=['Duplicate check', 'Count'],
)

tables = {
    '01_dataset_summary.csv': summary_table,
    '01_column_names.csv': column_names_table,
    '01_column_schema.csv': column_schema_table,
    '01_target_counts.csv': target_counts_table,
    '01_duplicate_summary.csv': duplicate_table,
    '01_proposal_verification.csv': proposal_table,
}

for filename, table in tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

print(f'Saved {len(tables)} verification tables to {TABLES_DIR.relative_to(PROJECT_ROOT)}')

Saved 6 verification tables to reports\tables


## T01 boundary

Dataset verification ends here. T02 Data Understanding has not started.